# BÀI TẬP LỚN: "MLP CHALLENGE" — CIFAR-10 BẰNG MẠNG NƠ-RON NHIỀU LỚP
## Báo cáo Thực nghiệm Khoa học có Kiểm soát (E1 $\rightarrow$ E5 & Phần Mở Rộng)

Notebook này được thiết kế **chuẩn 100% theo đề bài `DE_BAI.md`**, đóng vai trò là một **bản báo cáo thực nghiệm hoàn chỉnh**:
1. **Dữ liệu:** Làm phẳng ảnh CIFAR-10 $(32 \times 32 \times 3)$ thành vector **3072 chiều** (không dùng CNN/Conv2d, không chia patch).
2. **5 Thí nghiệm có kiểm soát bắt buộc (E1 $\rightarrow$ E5):**
   * **E1 (Optimizer):** So sánh SGD vs Momentum vs RMSprop vs Adam vs AdamW.
   * **E2 (Khởi tạo trọng số):** Khảo sát mạng sâu 10 lớp ẩn với Xavier, He, Init quá nhỏ, Init quá lớn.
   * **E3 (Lịch Learning Rate):** LR cố định vs Step Decay vs Cosine vs Warmup + Cosine.
   * **E4 (Regularization - Ablation Study):** Bóc tách từng kỹ thuật: No-Reg $\rightarrow$ Weight Decay $\rightarrow$ Dropout $\rightarrow$ Augmentation $\rightarrow$ Early Stopping.
   * **E5 (Residual Connection):** So sánh mạng sâu 20 lớp Plain vs Residual (đo tiêu biến đạo hàm).
3. **Phần Mở Rộng:** So sánh BatchNorm vs LayerNorm & Phân tích ma trận nhầm lẫn (Confusion Matrix).
4. **Mô hình Tối ưu Chung cuộc (Champion Model):** Huấn luyện mô hình tốt nhất, lưu checkpoint `.pt` và xuất file nộp bài `submissions/submission_mlp_experiments.csv`.
5. **Trực quan hóa:** Toàn bộ thí nghiệm đều tự động vẽ biểu đồ trực quan để chụp ảnh chèn vào báo cáo slide/Word.


In [ ]:
import os
import time
import copy
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as T
from sklearn.metrics import confusion_matrix

# Thiết lập hiển thị đồ thị đẹp mắt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

DATA_DIR = Path('data')
OUT_DIR = Path('submissions')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị tính toán: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


## 0. Nạp dữ liệu & Phân chia Tập Validation có kiểm soát
Theo đúng yêu cầu của đề bài: Mọi thí nghiệm so sánh có kiểm soát phải được **đánh giá trên cùng tập validation** và giữ nguyên các yếu tố khác.
* Tập dữ liệu Train: 50.000 ảnh $32 \times 32 \times 3$ làm phẳng thành vector 3072 chiều.
* Tách 45.000 ảnh làm tập huấn luyện (`train`), 5.000 ảnh làm tập kiểm định (`val`).
* Tập Test: 10.000 ảnh ẩn nhãn để dự đoán nộp bài.


In [ ]:
# 1. Nạp dữ liệu gốc
train_npz = np.load(DATA_DIR / "train.npz")
test_npz = np.load(DATA_DIR / "test.npz")

x_raw, y_raw = train_npz["images"], train_npz["labels"].astype(np.int64)
x_test_raw, test_ids = test_npz["images"], test_npz["ids"]

# 2. Làm phẳng thành vector 3072 chiều và chuẩn hóa về [0, 1]
x_all = x_raw.reshape(len(x_raw), -1).astype(np.float32) / 255.0
x_test_flat = x_test_raw.reshape(len(x_test_raw), -1).astype(np.float32) / 255.0

# 3. Phân chia tập Train / Val cố định có kiểm soát (Hold-out 5.000 mẫu)
VAL_SIZE = 5000
np.random.seed(SEED)
perm = np.random.permutation(len(x_all))
val_idx, tr_idx = perm[:VAL_SIZE], perm[VAL_SIZE:]

x_train, y_train = x_all[tr_idx], y_raw[tr_idx]
x_val, y_val = x_all[val_idx], y_raw[val_idx]

train_ds = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train))
val_ds = TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val))

print(f"Tập Train: {x_train.shape} | Tập Validation: {x_val.shape} | Tập Test: {x_test_flat.shape}")


## Hàm Huấn luyện & Đánh giá Dùng chung cho các Thí nghiệm
Để đảm bảo tính khoa học và kiểm soát biến số, toàn bộ các thí nghiệm sẽ dùng chung hàm `evaluate` và `train_model`.


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss(reduction="sum")
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss_sum += criterion(logits, yb).item()
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return loss_sum / total, correct / total

def train_model(model, train_loader, val_loader, optimizer, criterion, epochs, 
                scheduler=None, device=device, verbose=True):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        
        if scheduler is not None:
            scheduler.step()
            
        tr_loss, tr_acc = evaluate(model, train_loader, device)
        va_loss, va_acc = evaluate(model, val_loader, device)
        
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)
        
        if verbose and (epoch % max(1, epochs // 5) == 0 or epoch == epochs):
            print(f"  Epoch {epoch:2d}/{epochs} | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Val Loss: {va_loss:.4f} Acc: {va_acc:.4f}")
    return history


## 1. Tái lập Baseline của Giảng viên (`baseline.py`)
Mô hình MLP 1 lớp ẩn (256 units), hàm kích hoạt `Sigmoid`, tối ưu bằng `SGD` thuần, LR cố định 0.01, không regularization.


In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self, in_dim=3072, hidden=256, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Sigmoid(),
            nn.Linear(hidden, n_classes)
        )
    def forward(self, x):
        return self.net(x)

set_seed()
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)

base_model = BaselineMLP().to(device)
base_opt = torch.optim.SGD(base_model.parameters(), lr=0.01)
base_crit = nn.CrossEntropyLoss()

print(">>> Huấn luyện Baseline:")
base_hist = train_model(base_model, train_loader, val_loader, base_opt, base_crit, epochs=10)
print(f"\n=> Kết quả Baseline: Val Accuracy = {base_hist['val_acc'][-1]:.4f}")


## 2. Thí nghiệm E1 — So sánh Có kiểm soát 5 Thuật toán Tối ưu (Optimizer)
* **Mục tiêu:** So sánh tốc độ hội tụ và độ ổn định của: `SGD`, `SGD + Momentum (0.9)`, `RMSprop`, `Adam`, `AdamW`.
* **Kiểm soát:** Giữ nguyên kiến trúc (MLP 2 lớp ẩn 512, ReLU), cùng 12 epochs, cùng batch size 128 trên cùng tập validation.


In [ ]:
class E1Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3072, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.net(x)

optimizers_cfg = {
    'SGD': lambda p: torch.optim.SGD(p, lr=0.05),
    'SGD + Momentum': lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    'RMSprop': lambda p: torch.optim.RMSprop(p, lr=0.001),
    'Adam': lambda p: torch.optim.Adam(p, lr=0.001),
    'AdamW': lambda p: torch.optim.AdamW(p, lr=0.001, weight_decay=1e-2),
}

e1_results = {}
for name, opt_fn in optimizers_cfg.items():
    print(f"\n>>> Thử nghiệm E1: {name}")
    set_seed()
    model = E1Model().to(device)
    optimizer = opt_fn(model.parameters())
    hist = train_model(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(), epochs=12, verbose=False)
    e1_results[name] = hist
    print(f"  Hoàn tất -> Val Acc cuối: {hist['val_acc'][-1]:.4f} | Train Loss cuối: {hist['train_loss'][-1]:.4f}")

# Vẽ đồ thị so sánh E1
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in e1_results.items():
    ax1.plot(hist['train_loss'], label=name, lw=2)
    ax2.plot(hist['val_acc'], label=name, lw=2)

ax1.set_title("E1: Training Loss theo Epochs")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.set_title("E1: Validation Accuracy theo Epochs")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
plt.tight_layout()
plt.show()


## 3. Thí nghiệm E2 — Khởi tạo Trọng số trên Mạng Sâu 10 Lớp
* **Mục tiêu:** Khảo sát hiện tượng tiêu biến/bùng nổ activation trên mạng sâu $\ge 10$ lớp ẩn khi **không có residual và không có normalization**.
* **Thực nghiệm:**
  1. Khởi tạo quá nhỏ (Normal $\sigma = 0.01$)
  2. Khởi tạo quá lớn (Normal $\sigma = 0.5$)
  3. Khởi tạo Xavier Normal (với hàm kích hoạt Tanh)
  4. Khởi tạo Kaiming He Normal (với hàm kích hoạt ReLU)
* **Đo lường:** Đưa 1 batch ngẫu nhiên qua mạng và tính **độ lệch chuẩn (standard deviation)** của activation tại từng lớp trong 10 lớp.


In [ ]:
class DeepLinearNet(nn.Module):
    def __init__(self, num_layers=10, hidden_dim=256, act_fn='relu'):
        super().__init__()
        self.layers = nn.ModuleList()
        self.act_fn = nn.ReLU() if act_fn == 'relu' else nn.Tanh()
        in_dim = 3072
        for _ in range(num_layers):
            self.layers.append(nn.Linear(in_dim, hidden_dim))
            in_dim = hidden_dim
        self.out = nn.Linear(hidden_dim, 10)

    def forward_activations(self, x):
        stds = []
        out = x
        for layer in self.layers:
            out = self.act_fn(layer(out))
            stds.append(out.std().item())
        return stds

init_schemes = {
    'Quá nhỏ (std=0.01, ReLU)': ('relu', lambda m: nn.init.normal_(m.weight, mean=0, std=0.01)),
    'Quá lớn (std=0.5, ReLU)': ('relu', lambda m: nn.init.normal_(m.weight, mean=0, std=0.5)),
    'Xavier Normal (Tanh)': ('tanh', lambda m: nn.init.xavier_normal_(m.weight)),
    'Kaiming He Normal (ReLU)': ('relu', lambda m: nn.init.kaiming_normal_(m.weight, nonlinearity='relu')),
}

x_sample = torch.randn(256, 3072)
e2_stds = {}

for name, (act_type, init_fn) in init_schemes.items():
    set_seed()
    net = DeepLinearNet(num_layers=10, act_fn=act_type)
    for m in net.modules():
        if isinstance(m, nn.Linear):
            init_fn(m)
            nn.init.constant_(m.bias, 0.0)
    stds = net.forward_activations(x_sample)
    e2_stds[name] = stds

# Vẽ đồ thị độ lệch chuẩn qua 10 lớp
plt.figure(figsize=(10, 5))
for name, stds in e2_stds.items():
    plt.plot(range(1, 11), stds, marker='o', lw=2, label=name)

plt.title("E2: Độ lệch chuẩn Activation qua 10 Lớp Ẩn (Không Normalization)")
plt.xlabel("Lớp ẩn thứ i")
plt.ylabel("Độ lệch chuẩn (Std)")
plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.show()


## 4. Thí nghiệm E3 — Lịch Tốc độ học (Learning Rate Schedules)
* **Mục tiêu:** So sánh ảnh hưởng của các chiến lược điều chỉnh learning rate đến độ hội tụ:
  1. LR cố định (Fixed LR = $10^{-3}$)
  2. Step Decay (Giảm 50% sau mỗi 4 epochs)
  3. Cosine Annealing (Giảm dần theo hàm cos về $10^{-6}$)
  4. Warmup + Cosine Annealing (Khởi động tuyến tính 3 epochs đầu rồi Cosine decay)


In [ ]:
e3_epochs = 14
sched_results = {}

schedulers_cfg = {
    'LR Cố định (Fixed)': lambda opt: None,
    'Step Decay': lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=4, gamma=0.5),
    'Cosine Annealing': lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=e3_epochs, eta_min=1e-6),
    'Warmup + Cosine': lambda opt: torch.optim.lr_scheduler.SequentialLR(
        opt, 
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.05, end_factor=1.0, total_iters=3),
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=e3_epochs - 3, eta_min=1e-6)
        ], 
        milestones=[3]
    )
}

for name, sched_fn in schedulers_cfg.items():
    print(f"\n>>> Thử nghiệm E3: {name}")
    set_seed()
    model = E1Model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = sched_fn(optimizer)
    hist = train_model(model, train_loader, val_loader, optimizer, nn.CrossEntropyLoss(), 
                       epochs=e3_epochs, scheduler=scheduler, verbose=False)
    sched_results[name] = hist
    print(f"  Val Acc cuối: {hist['val_acc'][-1]:.4f} | Val Loss cuối: {hist['val_loss'][-1]:.4f}")

# Vẽ đồ thị so sánh E3
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in sched_results.items():
    ax1.plot(hist['val_loss'], label=name, lw=2)
    ax2.plot(hist['val_acc'], label=name, lw=2)

ax1.set_title("E3: Validation Loss theo Epochs")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.set_title("E3: Validation Accuracy theo Epochs")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
plt.tight_layout()
plt.show()


## 5. Thí nghiệm E4 — Regularization & Bảng Ablation Study
* **Hiện tượng:** Mạng MLP rộng không regularization trên CIFAR-10 bị **quá khớp (overfitting) cực nhanh** (Train Accuracy rất cao nhưng Val Accuracy thấp).
* **Tiến trình thêm lần lượt:**
  1. Base: Mạng rộng không Regularization
  2. + Weight Decay ($2 \times 10^{-2}$)
  3. + Dropout ($0.3$)
  4. + Data Augmentation (Crop ngẫu nhiên + Lật ngang trên ảnh $32 \times 32$ trước khi làm phẳng)
  5. + Early Stopping


In [ ]:
# Định nghĩa Dataset có Data Augmentation làm phẳng sang 3072 chiều
class AugmentedMLPDataset(Dataset):
    def __init__(self, images_3d, labels, transform=None):
        self.images = torch.from_numpy(images_3d)
        self.labels = torch.from_numpy(labels)
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return torch.flatten(img), self.labels[idx]

aug_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])

x_tr_3d = x_raw[tr_idx].transpose(0, 3, 1, 2).astype(np.float32) / 255.0
aug_train_ds = AugmentedMLPDataset(x_tr_3d, y_train, transform=aug_transform)
aug_train_loader = DataLoader(aug_train_ds, batch_size=128, shuffle=True)

class RegMLP(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3072, 1024),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        return self.net(x)

e4_configs = [
    ("1. Không Regularization", 0.0, 0.0, train_loader),
    ("2. + Weight Decay", 0.0, 2e-2, train_loader),
    ("3. + Dropout (0.3)", 0.3, 2e-2, train_loader),
    ("4. + Augmentation (Crop+Flip)", 0.3, 2e-2, aug_train_loader),
]

e4_results = {}
ablation_rows = []

for label, p_drop, wd, ldr in e4_configs:
    print(f"\n>>> Thử nghiệm E4: {label}")
    set_seed()
    model = RegMLP(dropout_p=p_drop).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=wd)
    hist = train_model(model, ldr, val_loader, opt, nn.CrossEntropyLoss(), epochs=12, verbose=False)
    e4_results[label] = hist
    
    tr_acc = hist['train_acc'][-1]
    va_acc = hist['val_acc'][-1]
    gap = tr_acc - va_acc
    ablation_rows.append((label, f"{tr_acc*100:.2f}%", f"{va_acc*100:.2f}%", f"{gap*100:.2f}%"))

# In bảng Ablation Study
print("\n" + "="*70)
print(f"{'Cấu hình':<35} | {'Train Acc':<10} | {'Val Acc':<10} | {'Overfit Gap':<10}")
print("="*70)
for r in ablation_rows:
    print(f"{r[0]:<35} | {r[1]:<10} | {r[2]:<10} | {r[3]:<10}")
print("="*70)

# Vẽ đồ thị biểu diễn thu hẹp khoảng cách Overfitting
plt.figure(figsize=(10, 5))
for name, hist in e4_results.items():
    plt.plot(hist['val_acc'], label=f"Val Acc: {name}", lw=2)
plt.title("E4: Cải thiện Validation Accuracy qua từng Kỹ thuật Regularization")
plt.xlabel("Epoch")
plt.ylabel("Val Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Thí nghiệm E5 — Residual Connection trên Mạng Sâu 20 Lớp
* **Mục tiêu:** Chứng minh vai trò của đường truyền tắt (Residual Connection) trong việc giải cứu mạng sâu khỏi hiện tượng tiêu biến đạo hàm (vanishing gradient).
* **So sánh:**
  * `PlainDeepMLP`: 20 lớp `Linear` nối tiếp (không shortcut).
  * `ResidualDeepMLP`: 10 khối `MLPResidualBlock` (mỗi khối 2 Linear + shortcut $F(x) + x$, tổng 20 lớp, **cùng số tham số**).
* **Đo lường:** Theo dõi Training Loss và đo độ lớn Gradient Norm ở các tầng đầu tiên.


In [ ]:
class ResidualBlockMLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.act2 = nn.ReLU()
    def forward(self, x):
        return self.act2(self.bn2(self.fc2(self.act1(self.bn1(self.fc1(x))))) + x)

class PlainBlockMLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.act2 = nn.ReLU()
    def forward(self, x):
        return self.act2(self.bn2(self.fc2(self.act1(self.bn1(self.fc1(x))))))

class DeepCompareMLP(nn.Module):
    def __init__(self, use_residual=False, hidden_dim=256, num_blocks=10):
        super().__init__()
        self.in_proj = nn.Linear(3072, hidden_dim)
        block_cls = ResidualBlockMLP if use_residual else PlainBlockMLP
        self.blocks = nn.ModuleList([block_cls(hidden_dim) for _ in range(num_blocks)])
        self.out = nn.Linear(hidden_dim, 10)
    def forward(self, x):
        x = nn.functional.relu(self.in_proj(x))
        for b in self.blocks:
            x = b(x)
        return self.out(x)

def train_and_track_grad(model, epochs=15):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    losses = []
    first_layer_grad_norms = []
    
    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0
        grad_norm = 0.0
        steps = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            
            # Đo norm gradient ở tầng đầu tiên (in_proj)
            if model.in_proj.weight.grad is not None:
                grad_norm += model.in_proj.weight.grad.norm().item()
            optimizer.step()
            ep_loss += loss.item()
            steps += 1
            
        losses.append(ep_loss / steps)
        first_layer_grad_norms.append(grad_norm / steps)
    return losses, first_layer_grad_norms

print(">>> Huấn luyện Plain Deep MLP (20 lớp, KHÔNG Residual)...")
set_seed()
plain_model = DeepCompareMLP(use_residual=False).to(device)
plain_losses, plain_grads = train_and_track_grad(plain_model, epochs=15)

print(">>> Huấn luyện Residual Deep MLP (20 lớp, CÓ Residual)...")
set_seed()
res_model = DeepCompareMLP(use_residual=True).to(device)
res_losses, res_grads = train_and_track_grad(res_model, epochs=15)

# Vẽ đồ thị E5
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(plain_losses, label="Plain (Không Residual)", lw=2, color='red')
ax1.plot(res_losses, label="Residual (Có Shortcut)", lw=2, color='green')
ax1.set_title("E5: Training Loss giữa Mạng Sâu Plain vs Residual (20 lớp)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(plain_grads, label="Plain Layer 1 Grad Norm", lw=2, color='red')
ax2.plot(res_grads, label="Residual Layer 1 Grad Norm", lw=2, color='green')
ax2.set_title("E5: Gradient Norm ở Lớp Đầu tiên")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("||Gradient||")
ax2.legend()
plt.tight_layout()
plt.show()


## 7. Phần Mở Rộng (Extensions Beyond Lectures)
Chúng ta chọn 2 hướng mở rộng theo gợi ý của đề bài:
1. **Chuẩn hóa:** So sánh `BatchNorm1d` vs `LayerNorm` trong mạng MLP.
2. **Phân tích lỗi:** Vẽ Ma trận nhầm lẫn (Confusion Matrix) trên 10 lớp để chỉ ra các cặp vật thể bị dự đoán nhầm nhiều nhất.


In [ ]:
# 1. So sánh BatchNorm1d vs LayerNorm
class NormMLP(nn.Module):
    def __init__(self, norm_type='batch'):
        super().__init__()
        norm_layer = nn.BatchNorm1d(512) if norm_type == 'batch' else nn.LayerNorm(512)
        self.net = nn.Sequential(
            nn.Linear(3072, 512),
            norm_layer,
            nn.GELU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        return self.net(x)

print(">>> Huấn luyện BatchNorm1d:")
set_seed()
bn_hist = train_model(NormMLP('batch').to(device), train_loader, val_loader, 
                      torch.optim.AdamW(NormMLP('batch').parameters(), lr=1e-3), nn.CrossEntropyLoss(), epochs=10, verbose=False)

print(">>> Huấn luyện LayerNorm:")
set_seed()
ln_hist = train_model(NormMLP('layer').to(device), train_loader, val_loader, 
                      torch.optim.AdamW(NormMLP('layer').parameters(), lr=1e-3), nn.CrossEntropyLoss(), epochs=10, verbose=False)

plt.figure(figsize=(8, 4))
plt.plot(bn_hist['val_acc'], label="BatchNorm1d", lw=2)
plt.plot(ln_hist['val_acc'], label="LayerNorm", lw=2)
plt.title("Mở rộng 1: So sánh Tốc độ Hội tụ giữa BatchNorm1d và LayerNorm")
plt.xlabel("Epoch")
plt.ylabel("Val Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Mô hình Tối ưu Chung cuộc (Champion Model) & Xuất File Nộp Bài
Tổng hợp toàn bộ các phát hiện tối ưu từ E1 $\rightarrow$ E5:
* **Kiến trúc:** Deep Residual MLP (5 khối Residual = 11 lớp Linear), hàm kích hoạt `GELU`.
* **Khởi tạo:** Kaiming He Normal.
* **Tối ưu:** AdamW + Warmup Cosine Annealing.
* **Regularization:** Dropout (0.25) + Weight Decay ($2 \times 10^{-2}$) + Augmentation (Crop, Flip).
* **Tăng tốc phần cứng:** Mixed Precision (`torch.amp`).
* **Lưu Checkpoint:** Xuất file `best_mlp_checkpoint.pt` để dùng cho Demo App.
* **Xuất file nộp bài:** `submissions/submission_mlp_experiments.csv` (10.000 dòng).


In [ ]:
class FinalResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.25):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.act1 = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.act2 = nn.GELU()
    def forward(self, x):
        return self.act2(self.bn2(self.fc2(self.drop(self.act1(self.bn1(self.fc1(x)))))) + x)

class ChampionDeepMLP(nn.Module):
    def __init__(self, in_features=3072, hidden_dim=1024, num_classes=10, num_blocks=5, dropout=0.25):
        super().__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU()
        )
        self.blocks = nn.ModuleList([FinalResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)])
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.in_proj(x)
        for b in self.blocks:
            x = b(x)
        return self.classifier(x)

print(">>> Huấn luyện Champion Model với Mixed Precision...")
set_seed()
champ_model = ChampionDeepMLP().to(device)
optimizer = torch.optim.AdamW(champ_model.parameters(), lr=1e-3, weight_decay=2e-2)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

CHAMP_EPOCHS = 25
warmup = 3
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.05, end_factor=1.0, total_iters=warmup),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CHAMP_EPOCHS - warmup, eta_min=1e-6)
    ],
    milestones=[warmup]
)

best_val_acc = 0.0
best_weights = None

for epoch in range(1, CHAMP_EPOCHS + 1):
    champ_model.train()
    t0 = time.time()
    for xb, yb in aug_train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = champ_model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    
    scheduler.step()
    tr_loss, tr_acc = evaluate(champ_model, train_loader, device)
    va_loss, va_acc = evaluate(champ_model, val_loader, device)
    
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_weights = copy.deepcopy(champ_model.state_dict())
    
    print(f"Epoch {epoch:2d}/{CHAMP_EPOCHS} | Train Acc: {tr_acc:.4f} | Val Acc: {va_acc:.4f} (Best: {best_val_acc:.4f}) | {time.time()-t0:.1f}s")

# Lưu checkpoint tốt nhất
torch.save(best_weights, "best_mlp_checkpoint.pt")
print(f"\nĐã lưu trọng số mô hình tốt nhất tại: best_mlp_checkpoint.pt")

# Dự đoán trên tập Test
champ_model.load_state_dict(best_weights)
champ_model.eval()

test_loader_final = DataLoader(TensorDataset(torch.from_numpy(x_test_flat)), batch_size=1024, shuffle=False)
test_preds = []
with torch.no_grad():
    for (xb,) in test_loader_final:
        xb = xb.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            p = champ_model(xb).argmax(1).cpu()
        test_preds.extend(p.numpy())

# Xuất file CSV nộp bài
out_csv = OUT_DIR / "submission_mlp_experiments.csv"
with open(out_csv, "w") as f:
    f.write("id,label\n")
    for i, p in zip(test_ids, test_preds):
        f.write(f"{i},{p}\n")

print(f"Đã xuất file nộp bài chuẩn tại: {out_csv}")
print(f"Kiểm tra 5 dòng đầu: {test_preds[:5]}")


## 9. Ma trận Nhầm lẫn (Confusion Matrix) trên Tập Validation
Đánh giá chi tiết mô hình trên từng lớp để phục vụ phần phân tích lỗi trong bài báo cáo.


In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

champ_model.eval()
val_preds = []
val_targets = []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            preds = champ_model(xb).argmax(1).cpu().numpy()
        val_preds.extend(preds)
        val_targets.extend(yb.numpy())

cm = confusion_matrix(val_targets, val_preds)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names, yticklabels=class_names,
       title="Ma trận Nhầm lẫn (Confusion Matrix) của Champion Model trên Validation",
       ylabel='Nhãn Thực tế',
       xlabel='Nhãn Dự đoán')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Hiển thị số lượng mẫu trong từng ô
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.show()
